# Run Inference on FULL MathFish Dev Set (1,942 problems)

**IMPORTANT: Set Runtime to GPU (T4 or A100)**
- Runtime → Change runtime type → Hardware accelerator: GPU

In [ ]:
# 1. Check GPU
!nvidia-smi

In [ ]:
# 2. Clone repository from GitHub
!git clone https://github.com/matrix-mayank/mathfish.git
%cd mathfish

In [ ]:
# 3. Install dependencies
!pip install -q torch transformers sentence-transformers datasets huggingface-hub tqdm

In [ ]:
# 4. Download FULL dev dataset (1,942 labeled problems)
!python scripts/sample_dev.py --n 999999 --output data/processed/problems_dev_full.jsonl

# Check
!wc -l data/processed/problems_dev_full.jsonl
!echo "Expected: 1942 problems"

In [ ]:
# 5. Upload trained models (zip files)
# You need: biencoder_full_15k.zip and verifier.zip
from google.colab import files
import zipfile
import os

print("Upload biencoder_full_15k.zip (or biencoder_5k_gpu.zip):")
uploaded = files.upload()

for filename in uploaded.keys():
    print(f'Extracting {filename}...')
    with zipfile.ZipFile(filename, 'r') as zip_ref:
        # Extract to appropriate folder based on filename
        if 'full_15k' in filename:
            os.makedirs('outputs/biencoder_full_15k/checkpoint', exist_ok=True)
            zip_ref.extractall('outputs/biencoder_full_15k/checkpoint')
        else:
            os.makedirs('outputs/biencoder_5k_gpu/checkpoint', exist_ok=True)
            zip_ref.extractall('outputs/biencoder_5k_gpu/checkpoint')

print("\nNow upload verifier.zip:")
uploaded = files.upload()

for filename in uploaded.keys():
    print(f'Extracting {filename}...')
    with zipfile.ZipFile(filename, 'r') as zip_ref:
        os.makedirs('outputs/verifier', exist_ok=True)
        zip_ref.extractall('outputs/verifier')

In [ ]:
# 6. Verify models exist
import os

# Check which bi-encoder we have
if os.path.exists('outputs/biencoder_full_15k/checkpoint'):
    biencoder_path = 'outputs/biencoder_full_15k'
    print("✅ Using biencoder_full_15k")
else:
    biencoder_path = 'outputs/biencoder_5k_gpu'
    print("✅ Using biencoder_5k_gpu")

!ls -lh {biencoder_path}/checkpoint/ | head -10
!ls -lh outputs/verifier/ | head -10

In [ ]:
# 7. Run inference with threshold sweep
# This will test multiple thresholds and find the best one
# Expected time: ~15-20 minutes on A100, ~40-50 minutes on T4

import subprocess
import json

# Determine bi-encoder path
import os
if os.path.exists('outputs/biencoder_full_15k/checkpoint'):
    biencoder_path = 'outputs/biencoder_full_15k'
else:
    biencoder_path = 'outputs/biencoder_5k_gpu'

thresholds = [0.5, 0.7, 0.8, 0.85, 0.9, 0.95]
results = []

for t in thresholds:
    print(f"\n{'='*60}")
    print(f"Testing threshold {t}")
    print(f"{'='*60}")
    
    output_file = f"outputs/hybrid_dev_full_t{t}.jsonl"
    
    # Run inference
    result = subprocess.run([
        "python", "scripts/run_hybrid_rag.py",
        "--biencoder_path", biencoder_path,
        "--verifier_path", "outputs/verifier",
        "--problems_file", "data/processed/problems_dev_full.jsonl",
        "--output_file", output_file,
        "--top_k", "20",
        "--verification_threshold", str(t),
        "--device", "cuda"
    ], check=True, capture_output=True, text=True)
    
    # Print stats from inference
    print(result.stdout.split('\n')[-4:])
    
    # Evaluate
    eval_result = subprocess.run([
        "python", "scripts/evaluate_alignment.py",
        "--predictions_file", output_file,
        "--gold_file", "data/processed/problems_dev_full.jsonl"
    ], capture_output=True, text=True)
    
    metrics = json.loads(eval_result.stdout)
    metrics['threshold'] = t
    results.append(metrics)
    
    print(f"\n📊 Results:")
    print(f"  Micro F1: {metrics['micro_f1']:.4f}")
    print(f"  Macro F1: {metrics['macro_f1']:.4f}")
    print(f"  Recall@5: {metrics['recall@5']:.4f}")
    print(f"  Recall@10: {metrics['recall@10']:.4f}")
    print(f"  Recall@20: {metrics['recall@20']:.4f}")

# Summary table
print(f"\n{'='*80}")
print("📋 FINAL SUMMARY")
print(f"{'='*80}")
print(f"\n{'Threshold':<12} {'Micro F1':<12} {'Macro F1':<12} {'Recall@5':<12} {'Recall@10':<12} {'Recall@20':<12}")
print("-" * 80)
for r in results:
    print(f"{r['threshold']:<12} {r['micro_f1']:<12.4f} {r['macro_f1']:<12.4f} {r['recall@5']:<12.4f} {r['recall@10']:<12.4f} {r['recall@20']:<12.4f}")

# Find best threshold
best = max(results, key=lambda x: x['micro_f1'])
print(f"\n🎯 Best threshold: {best['threshold']} (Micro F1: {best['micro_f1']:.4f})")

In [ ]:
# 8. Download all results
import shutil
from google.colab import files

# Zip all prediction files
shutil.make_archive('inference_results_full_dev', 'zip', 'outputs')

# Download
files.download('inference_results_full_dev.zip')

print("\n✅ Download complete!")
print(f"\n📊 Evaluated on {1942} problems (full dev set)")
print(f"🎯 Best threshold: {best['threshold']}")
print(f"📈 Best Micro F1: {best['micro_f1']:.4f}")

## Summary

**What we evaluated:**
- Dataset: 1,942 labeled problems from MathFish dev split
- Thresholds tested: 0.5, 0.7, 0.8, 0.85, 0.9, 0.95
- Time: ~15-20 mins on A100, ~40-50 mins on T4

**Key metrics:**
- Best threshold will be printed above
- This is 19x more reliable than 100 problem evaluation
- Results should match paper baselines better

**Comparison:**
- Previous eval (100 problems): Best F1 = 9.52% @ threshold 0.9
- Current eval (1,942 problems): See results above!